# Model Training and Testing Only
This notebook is a minimal, self-contained pipeline that focuses on training and testing models for SPY forecasting.

In [1]:
# Imports
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import yfinance as yf

from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print('✅ Libraries ready')

✅ Libraries ready


In [2]:
# Config
START_DATE = '2010-01-01'
END_DATE = '2024-12-31'
PREDICTION_HORIZON = 1  # predict 1 day ahead

In [3]:
# Minimal data loading and feature engineering (technical indicators only)
def get_stock_data(symbol='SPY', start=START_DATE, end=END_DATE):
    data = yf.download(symbol, start=start, end=end, progress=False)
    if data is None or data.empty:
        raise RuntimeError(f'No data for {symbol}')
    return data

def create_technical_indicators(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()
    close = df['Close'].astype(float)
    open_ = df['Open'].astype(float)
    high = df['High'].astype(float)
    low = df['Low'].astype(float)
    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].astype(float)

    # Basic returns
    df['Returns'] = close.pct_change()

    # Moving averages
    for w in [5, 10, 20, 50, 200]:
        df[f'SMA_{w}'] = close.rolling(w, min_periods=1).mean()
        df[f'EMA_{w}'] = close.ewm(span=w, adjust=False).mean()
    
    # RSI
    def rsi(prices, window=14):
        delta = prices.diff()
        gain = delta.where(delta > 0, 0.0)
        loss = -delta.where(delta < 0, 0.0)
        avg_gain = gain.rolling(window=window, min_periods=1).mean()
        avg_loss = loss.rolling(window=window, min_periods=1).mean()
        rs = avg_gain / avg_loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))
    df['RSI'] = rsi(close)

    # Volatility
    df['Volatility_20'] = df['Returns'].rolling(20, min_periods=1).std()

    # Bollinger bands position
    sma20 = close.rolling(20, min_periods=1).mean()
    std20 = close.rolling(20, min_periods=1).std()
    bb_u = sma20 + 2*std20
    bb_l = sma20 - 2*std20
    df['BB_Position'] = (close - bb_l) / (bb_u - bb_l).replace(0, np.nan)

    # Momentum
    for p in [5, 10, 20]:
        df[f'Momentum_{p}'] = close / close.shift(p)

    return df

# Load dataset (prefer prepared CSV if present)
try:
    final_dataset = pd.read_csv('final_dataset.csv', index_col=0, parse_dates=True)
    print('📄 Loaded final_dataset.csv')
except Exception:
    print('⬇️  Downloading SPY and creating technical features...')
    spy = get_stock_data('SPY', START_DATE, END_DATE)
    final_dataset = create_technical_indicators(spy)

# Keep only numeric columns and drop rows with all-NaN
final_dataset = final_dataset.select_dtypes(include=[np.number]).dropna(how='all')
print('Data shape:', final_dataset.shape)

⬇️  Downloading SPY and creating technical features...
Data shape: (3773, 22)


In [4]:
# Prepare ML dataset (target = next-day price delta) and time-series split + scaling
def prepare_ml_dataset(df: pd.DataFrame, target_col='Delta', prediction_horizon=1, remove_leakage=True):
    ml = df.copy()
    if remove_leakage:
        for col in ['Open', 'High', 'Low', 'Close', 'Adj Close']:
            if col in ml.columns:
                for lag in [1,2,3,5]:
                    ml[f'{col}_lag{lag}'] = ml[col].shift(lag)
        drop_cols = [c for c in ['Open','High','Low','Close','Adj Close'] if c in ml.columns]
        ml = ml.drop(columns=drop_cols)
    # Construct target
    if 'Close' in df.columns:
        ml['Target'] = df['Close'].shift(-prediction_horizon) - df['Close']
    else:
        raise ValueError('Close column is required to compute Delta target')
    # Clean
    ml = ml.dropna()
    X = ml.drop(columns=['Target'])
    y = ml['Target']
    return X, y

def time_series_split(X, y, train_ratio=0.7, val_ratio=0.15):
    n = len(X)
    tr_end = int(n*train_ratio)
    va_end = int(n*(train_ratio+val_ratio))
    X_tr, y_tr = X.iloc[:tr_end], y.iloc[:tr_end]
    X_va, y_va = X.iloc[tr_end:va_end], y.iloc[tr_end:va_end]
    X_te, y_te = X.iloc[va_end:], y.iloc[va_end:]
    return X_tr, X_va, X_te, y_tr, y_va, y_te

# Build X, y
X, y = prepare_ml_dataset(final_dataset, target_col='Delta', prediction_horizon=PREDICTION_HORIZON)
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X, y)

# Scale features
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
print('Splits:', X_train.shape, X_val.shape, X_test.shape)

Splits: (2625, 34) (563, 34) (563, 34)


In [5]:
# Train classical ML models and generate predictions
def train_models(X_tr, y_tr, X_va, y_va):
    models = {}
    # Linear regularized
    ridge = Ridge(alpha=1.0, random_state=42); ridge.fit(X_tr, y_tr); models['Ridge'] = ridge
    lasso = Lasso(alpha=0.1, random_state=42, max_iter=2000); lasso.fit(X_tr, y_tr); models['Lasso'] = lasso
    # Ensembles
    rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr); models['Random Forest'] = rf
    gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42)
    gb.fit(X_tr, y_tr); models['Gradient Boosting'] = gb
    # Boosting libs
    xgb_model = xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42, n_jobs=-1)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False); models['XGBoost'] = xgb_model
    lgb_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42, verbose=-1, n_jobs=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]); models['LightGBM'] = lgb_model
    return models

trained_models = train_models(X_train_scaled, y_train, X_val_scaled, y_val)

# Predictions dict
predictions = {name: {
    'train': m.predict(X_train_scaled),
    'val':   m.predict(X_val_scaled),
    'test':  m.predict(X_test_scaled)
} for name, m in trained_models.items()}
print('✅ Classical models trained and predicted')

[LightGBM] [Fatal] Do not support special JSON characters in feature name.


LightGBMError: Do not support special JSON characters in feature name.

In [ ]:
# LSTM training and predictions
def create_lstm_sequences(X_df, y_s, sequence_length=30):
    X_seq, y_seq = [], []
    for i in range(sequence_length, len(X_df)):
        X_seq.append(X_df.iloc[i-sequence_length:i].values)
        y_seq.append(y_s.iloc[i])
    return np.array(X_seq), np.array(y_seq)

def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        BatchNormalization(), Dropout(0.3),
        LSTM(64, return_sequences=True),
        BatchNormalization(), Dropout(0.3),
        LSTM(32, return_sequences=False),
        BatchNormalization(), Dropout(0.2),
        Dense(64, activation='relu'), BatchNormalization(), Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    return model

seq_len = 30
X_train_seq, y_train_seq = create_lstm_sequences(X_train_scaled, y_train, seq_len)
X_val_seq,   y_val_seq   = create_lstm_sequences(X_val_scaled,   y_val,   seq_len)
X_test_seq,  y_test_seq  = create_lstm_sequences(X_test_scaled,  y_test,  seq_len)

lstm_model = build_lstm((seq_len, X_train_scaled.shape[1]))
early = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30, batch_size=32, callbacks=[early, reduce], verbose=0
)

lstm_pred_train = lstm_model.predict(X_train_seq, verbose=0).flatten()
lstm_pred_val   = lstm_model.predict(X_val_seq,   verbose=0).flatten()
lstm_pred_test  = lstm_model.predict(X_test_seq,  verbose=0).flatten()

# Append to predictions aligned to sequence start
predictions['LSTM'] = {
    'train': lstm_pred_train,
    'val':   lstm_pred_val,
    'test':  lstm_pred_test
}

# Align others by trimming first seq_len elements
predictions = {
    k: (v if k=='LSTM' else {
        'train': v['train'][seq_len:],
        'val':   v['val'][seq_len:],
        'test':  v['test'][seq_len:]
    }) for k, v in predictions.items()
}

y_train_adj = y_train.iloc[seq_len:]
y_val_adj   = y_val.iloc[seq_len:]
y_test_adj  = y_test.iloc[seq_len:]
print('✅ LSTM trained and predictions added')

In [ ]:
# Evaluation (RMSE, MAE, R2, Directional Accuracy)
def evaluate(preds: dict, y_true, name='Test'):
    res = {}
    yt = pd.Series(np.asarray(y_true).reshape(-1)).astype(float).reset_index(drop=True)
    print(f'
{name} set:')
    for model_name, arr in preds.items():
        yp = pd.Series(np.asarray(arr).reshape(-1)).astype(float)
        n = min(len(yt), len(yp))
        yt_n, yp_n = yt.iloc[:n], yp.iloc[:n]
        rmse = np.sqrt(mean_squared_error(yt_n, yp_n))
        mae  = mean_absolute_error(yt_n, yp_n)
        r2   = r2_score(yt_n, yp_n)
        # directional accuracy for deltas
        acc = (np.sign(yt_n) == np.sign(yp_n)).mean()
        res[model_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'Directional_Accuracy': acc}
        print(f
)
    return pd.DataFrame(res).T

test_results = evaluate({k:v['test'] for k,v in predictions.items()}, y_test_adj, 'Test')
val_results  = evaluate({k:v['val']  for k,v in predictions.items()}, y_val_adj,  'Validation')
test_results, val_results